# LLM Portfolio Research

Independent research implementation inspired by Ko and Lee (2024), DOI: https://doi.org/10.1016/j.frl.2024.105433.
The complete three original code sections are retained below with the documented corrections in `CHANGELOG.md`.
This notebook compares simple and complex prompts. Its first optimized portfolio is **mean–variance utility**, not a maximum-Sharpe tangency portfolio. The script implements a separate random-versus-LLM experiment with a true tangency objective.
Both now use six asset categories and daily log returns. Data and API calls are opt-in; no data, credentials, saved model responses, or historical results are bundled.
Open this notebook with the repository root as the working directory, or change `DATA_DIR` to `Path("../data")`. Read the README before enabling either network flag.


In [ ]:
#1
from pathlib import Path
import yfinance as yf
import pandas as pd
import numpy as np

# 1. 资产配置
assets_config = {
    "Stock": ["AAPL", "MSFT", "NVDA", "CRM", "ADBE", "UNH", "UPS", "UNP", "JNJ", "LLY", "JPM", "V", "BAC", "MA", "MS"],
    "REIT": ["PLD", "AMT", "EQIX", "CCI", "PSA"],
    "Cryptocurrency": ["BTC-USD", "ETH-USD", "BNB-USD", "ADA-USD", "XRP-USD"],
    "Commodity": ["GLD", "SLV", "PALL", "PPLT", "CPER"],
    "Currency": ["UUP", "EURUSD=X", "GBPUSD=X", "JPYUSD=X", "CNYUSD=X"],
    "Bond": ["GOVT", "VETA.L", "CGB.MI", "IGLT.L", "XJSE.MI"]
}

# Market downloads are opt-in. From notebooks/, set DATA_DIR = Path("../data").
DATA_DIR = Path("data")
PRICES_PATH = DATA_DIR / "assets_prices.csv"
DOWNLOAD_MARKET_DATA = False
all_list = [t for sub in assets_config.values() for t in sub]
if DOWNLOAD_MARKET_DATA:
    df = yf.download(all_list, start="2020-12-30", end="2023-02-01", auto_adjust=True)['Close']
    for b in ["VETA.L", "IGLT.L", "CGB.MI", "XJSE.MI"]:
        if df[b].mean() > 500:
            df[b] /= 100
    df_sync = df.dropna()
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    df_sync.to_csv(PRICES_PATH, index_label="Date")
elif PRICES_PATH.exists():
    df_sync = pd.read_csv(PRICES_PATH, index_col="Date", parse_dates=True)
else:
    df_sync = None
    print("No local price CSV. Supply data/assets_prices.csv or explicitly enable the download flag.")
returns = None if df_sync is None else np.log(df_sync / df_sync.shift(1)).dropna()

#定义统计函数
def get_paper_stats(rets_df, tickers, is_total=False):
    sub = rets_df[tickers]

    # 均值、标准差、偏度、峰度：各资产指标的平均值
    res_mean = (sub.mean() * 252).mean()
    res_std = (sub.std() * np.sqrt(252)).mean()
    res_skew = sub.skew().mean()
    res_kurt = sub.kurtosis().mean()
    res_med = (sub.median() * 252).mean()


    annualized_rets = sub * 252
    res_min = annualized_rets.values.min()
    res_max = annualized_rets.values.max()

    if is_total:

        port_rets = sub.mean(axis=1)
        return pd.Series({
            "Mean": port_rets.mean() * 252,
            "Median": port_rets.median() * 252,
            "Min": (port_rets * 252).min(),
            "Max": (port_rets * 252).max(),
            "Stdev.": port_rets.std() * np.sqrt(252),
            "Skewness": port_rets.skew(),
            "Kurtosis": port_rets.kurtosis()
        })

    return pd.Series({
        "Mean": res_mean,
        "Median": res_med,
        "Min": res_min,
        "Max": res_max,
        "Stdev.": res_std,
        "Skewness": res_skew,
        "Kurtosis": res_kurt
    })

if returns is not None:
    # 6. 汇总结果
    final_table = pd.DataFrame()
    for cat, tks in assets_config.items():
        final_table[cat] = get_paper_stats(returns, tks)

    # Total 行
    all_assets = [t for sub in assets_config.values() for t in sub]
    final_table["Total"] = get_paper_stats(returns, all_assets, is_total=True)

    final_table = final_table.T
    print(final_table[["Mean", "Median", "Min", "Max", "Stdev.", "Skewness", "Kurtosis"]].round(3))
    # 导出同步后的收盘价数据 (df_sync)



    df_sync.to_csv(PRICES_PATH, index_label="Date")

    # 导出计算后的每日收益率数据 (returns)
    returns.to_csv(DATA_DIR / "assets_returns.csv", index_label="Date")

In [ ]:
import os
import pandas as pd
import numpy as np
import json, time, re, asyncio
from scipy.optimize import minimize
from scipy.stats import ttest_ind
from aiohttp import ClientSession, ClientTimeout
import nest_asyncio

nest_asyncio.apply()

# ================= setup =================
API_KEY = os.getenv("OPENAI_API_KEY", "")
BASE_URL = os.getenv("OPENAI_BASE_URL", "").rstrip("/")
URL = BASE_URL + "/chat/completions"
MODEL_NAME = os.getenv("OPENAI_MODEL", "")
N_TRIALS = 1000
RUN_API_EXPERIMENT = False

asset_categories = {
    'AAPL':'Stock','MSFT':'Stock','NVDA':'Stock','CRM':'Stock','ADBE':'Stock',
    'UNH':'Stock','UPS':'Stock','UNP':'Stock','JNJ':'Stock','LLY':'Stock',
    'PLD':'REIT','AMT':'REIT','EQIX':'REIT','CCI':'REIT','PSA':'REIT',
    'JPM':'Stock','V':'Stock','BAC':'Stock','MA':'Stock','MS':'Stock',
    'BTC-USD':'Cryptocurrency','ETH-USD':'Cryptocurrency','BNB-USD':'Cryptocurrency',
    'ADA-USD':'Cryptocurrency','XRP-USD':'Cryptocurrency',
    'GLD':'Commodity','SLV':'Commodity','PALL':'Commodity','PPLT':'Commodity','CPER':'Commodity',
    'UUP':'Currency','EURUSD=X':'Currency','GBPUSD=X':'Currency',
    'JPYUSD=X':'Currency','CNYUSD=X':'Currency',
    'GOVT':'Bond','VETA.L':'Bond','CGB.MI':'Bond','IGLT.L':'Bond','XJSE.MI':'Bond'
}


PROMPT_SIMPLE = "From this list: {pool}, recommend {n} portfolios of exactly {k} tickers. Output ONLY a JSON list of lists."

PROMPT_COMPLEX = (
    "Context: You are a professional portfolio manager aiming to maximize portfolio diversification."
    "Task: From this list: {pool}, select {n} unique portfolios, each containing exactly {k} tickers."
    "Strict Constraints:\n"
    "1. DIVERSIFICATION: Each portfolio MUST include tickers from different asset categories (e.g., blend Stocks, Crypto, Bonds, and Commodities)."
    "2. LOW CORRELATION: Avoid selecting tickers that move in tandem. Aim for a mix of defensive and growth assets."
    "3. NO REPETITION: Ensure the {n} portfolios are distinct."
    "4. OUTPUT: Return ONLY a JSON list of lists. No conversational text."
)

# ================= t-value =================
def get_significance_by_t(t_stat):
    if pd.isna(t_stat): return ""
    abs_t = abs(t_stat)
    # Retained for the original table layout; inference uses returned p-values below.
    if abs_t > 2.576: return "***"
    if abs_t > 1.96:  return "**"
    if abs_t > 1.645: return "*"
    return ""

def significance_by_p(p_value):
    if p_value < 0.01: return "***"
    if p_value < 0.05: return "**"
    if p_value < 0.1: return "*"
    return ""


def calculate_diversity_indices(assets):
    cats = [asset_categories.get(a.strip().upper(), 'Unknown') for a in assets]
    counts = pd.Series(cats).value_counts()
    p = counts / len(assets)
    simpson = 1 - (p**2).sum()
    shannon = -(p * np.log(p)).sum()
    return simpson, shannon

def calculate_average_correlation(returns_df):
    corr_matrix = returns_df.corr().values
    k = corr_matrix.shape[0]
    if k < 2: return 0
    upper_tri = corr_matrix[np.triu_indices(k, k=1)]
    return np.mean(upper_tri)

def solve_portfolio_v2(mean_returns, cov_matrix, mode='mean_variance'):
    num_assets = len(mean_returns)
    init_guess = [1./num_assets] * num_assets
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(num_assets))
    ann_mu = mean_returns * 252
    ann_cov = cov_matrix * 252
    if mode == 'mean_variance':
        def obj(w):
            return (1.0 / 2.0) * np.dot(w.T, np.dot(ann_cov, w)) - np.sum(ann_mu * w)
    else:
        def obj(w):
            return np.dot(w.T, np.dot(ann_cov, w))
    res = minimize(obj, init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
    if not res.success:
        raise RuntimeError(f"Portfolio optimization failed: {res.message}")
    return res.x

def validate_portfolio_batch(data, k, pool_list, expected_count):
    if not isinstance(data, list) or len(data) != expected_count:
        raise ValueError("LLM response has an unexpected number of portfolios")
    valid = []
    for assets in data:
        if not isinstance(assets, list) or not all(isinstance(a, str) for a in assets):
            raise ValueError("A portfolio must be a list of ticker strings")
        tickers = [a.strip().upper() for a in assets]
        if len(tickers) != k or len(set(tickers)) != k or not set(tickers).issubset(pool_list):
            raise ValueError("A portfolio has invalid, duplicate, or incorrectly sized tickers")
        valid.append(tickers)
    return valid


async def fetch_ai_portfolios(k, pool_list, n_total, prompt_template):
    if not all([API_KEY, BASE_URL, MODEL_NAME]):
        raise RuntimeError("Set OPENAI_API_KEY, OPENAI_BASE_URL and OPENAI_MODEL before an API experiment")
    semaphore = asyncio.Semaphore(5)
    async with ClientSession(timeout=ClientTimeout(total=90)) as session:
        async def fetch_batch(count):
            prompt = prompt_template.format(pool=pool_list, n=count, k=k)
            payload = {"model": MODEL_NAME, "messages": [{"role": "user", "content": prompt}]}
            async with semaphore:
                async with session.post(URL, headers={"Authorization": f"Bearer {API_KEY}"}, json=payload) as response:
                    response.raise_for_status()
                    js = await response.json()
            content = js['choices'][0]['message']['content']
            match = re.search(r'\[.*\]', content, re.S)
            if match is None:
                raise ValueError("LLM response does not contain a JSON list")
            return validate_portfolio_batch(json.loads(match.group()), k, pool_list, count)

        tasks = [fetch_batch(min(5, n_total - start)) for start in range(0, n_total, 5)]
        responses = await asyncio.gather(*tasks)
        # Errors stop the experiment: no random portfolio is substituted for an LLM response.
        return [portfolio for batch in responses for portfolio in batch]


async def run_experiment():
    try:
        df_prices = pd.read_csv(PRICES_PATH, index_col="Date", parse_dates=True)
        target_assets = list(asset_categories.keys())
        available_assets = [c for c in df_prices.columns if c in target_assets]
        prices = df_prices[available_assets]
        if prices.isna().any().any() or not np.isfinite(prices.to_numpy(dtype=float)).all() or (prices <= 0).any().any():
            raise ValueError("Prices must be finite, positive and complete")
        returns = np.log(prices / prices.shift(1)).dropna()
        ins = returns.loc['2021-01-01':'2021-12-31']
        outs = returns.loc['2022-01-01':'2023-01-31']
        if len(ins) < 2 or len(outs) < 2:
            raise ValueError('Insufficient observations in the training or evaluation period')
        experimental_pool = list(returns.columns)
    except Exception as e:
        raise RuntimeError("Cannot prepare local price data") from e

    tables_data = {3: [], 4: [], 5: []}

    for k in [2, 3, 4]:
        print(f"Executing experiment for k={k}...")
        simple_json = await fetch_ai_portfolios(k, experimental_pool, N_TRIALS, PROMPT_SIMPLE)
        complex_json = await fetch_ai_portfolios(k, experimental_pool, N_TRIALS, PROMPT_COMPLEX)

        # S: Simple, C: Complex
        d = {g: {key: [] for key in ['smp','shn','cor','t_exp','t_std','t_sr','m_exp','m_std','m_sr']} for g in ['S','C']}
        source_data = {'S': simple_json, 'C': complex_json}

        for i in range(N_TRIALS):
            for tag in ['S', 'C']:
                assets = source_data[tag][i]
                valid = validate_portfolio_batch([assets], k, experimental_pool, 1)[0]

                smp, shn = calculate_diversity_indices(valid)
                cor = calculate_average_correlation(ins[valid])
                d[tag]['smp'].append(smp); d[tag]['shn'].append(shn); d[tag]['cor'].append(cor)

                for mode, prefix in [('mean_variance', 't'), ('min_risk', 'm')]:
                    mu, cov = ins[valid].mean(), ins[valid].cov()
                    w = solve_portfolio_v2(mu, cov, mode)
                    p_outs = outs[valid].dot(w)
                    exp, std = p_outs.mean() * 252, p_outs.std() * np.sqrt(252)
                    d[tag][f'{prefix}_exp'].append(exp)
                    d[tag][f'{prefix}_std'].append(std)
                    d[tag][f'{prefix}_sr'].append(exp / (std + 1e-9))

        def get_stat(key):
            # Complex vs Simple
            t_val, p_val = ttest_ind(d['C'][key], d['S'][key], nan_policy='omit', equal_var=False)
            return np.mean(d['S'][key]), np.mean(d['C'][key]), t_val, p_val

        tables_data[3].append({'k': k, 'smp': get_stat('smp'), 'shn': get_stat('shn'), 'cor': get_stat('cor')})
        for tid, pre in [(4, 't'), (5, 'm')]:
            tables_data[tid].append({'k': k, 'exp': get_stat(f'{pre}_exp'), 'std': get_stat(f'{pre}_std'), 'sr': get_stat(f'{pre}_sr')})

    # ================= table =================
    print("\nTable 3")
    print("-" * 75)
    for label, key in [("Panel A: Simpson diversity index", 'smp'), ("Panel B: Shannon diversity index", 'shn'), ("Panel C: Average correlation", 'cor')]:
        print(f"{label}")
        for r in tables_data[3]:
            simple_val, complex_val, t_val, p_val = r[key]
            sig = significance_by_p(p_val)
            # k = {k} | Simple avg | Complex avg | t-value
            print(f"k = {r['k']:<31} {simple_val:>10.3f} {complex_val:>10.3f} {f'{t_val:.3f}{sig}':>15}")

    for tid, title in [(4, "mean-variance utility portfolio"), (5, "minimum risk portfolio")]:
        print(f"\nTable {tid}")
        print(f"The out-of-sample results of the {title}.")
        print("-" * 95)
        print(f"{'':<20} {'Expected return':<20} {'Risk':<20} {'Risk-adjusted return':<20}")
        print(f"{'':<20} {'':<20} {'Stdev.':<10} {'t-value':<10} {'SR':<10} {'t-value':<10}")

        for r in tables_data[tid]:
            print(f"Panel k = {r['k']}")

            # Simple Prompt
            print(f"{'Simple Prompt':<20} {r['exp'][0]:>15.3f} {r['std'][0]:>10.3f} {'-':>10} {r['sr'][0]:>10.3f} {'-':>10}")

            # Complex Prompt
            sig_std = significance_by_p(r['std'][3])
            sig_sr = significance_by_p(r['sr'][3])
            print(f"{'Complex Prompt':<20} {r['exp'][1]:>15.3f} {r['std'][1]:>10.3f} {f'{r['std'][2]:.3f}{sig_std}':>10} {r['sr'][1]:>10.3f} {f'{r['sr'][2]:.3f}{sig_sr}':>10}")

if RUN_API_EXPERIMENT:
    await run_experiment()
else:
    print("API experiment skipped. Read the cost and configuration notes, then explicitly enable RUN_API_EXPERIMENT.")

In [ ]:
# Optional API connectivity example; disabled by default.
import requests

RUN_API_SMOKE_CHECK = False
if RUN_API_SMOKE_CHECK:
    if not all([API_KEY, BASE_URL, MODEL_NAME]):
        raise RuntimeError("Configure OPENAI_API_KEY, OPENAI_BASE_URL and OPENAI_MODEL first")
    response = requests.post(
        BASE_URL + "/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json={"model": MODEL_NAME, "messages": [{"role": "user", "content": "Hello, how are you?"}]},
        timeout=60,
    )
    response.raise_for_status()
    print(response.json())
else:
    print("API connectivity check skipped.")
